In [63]:
import pandas as pd

DATA_DIR = "datasets/msk_chord_2024"  # relative to Clinical\datasets\
root_path = "datasets/msk_chord_2024/msk_chord_out"

patient = pd.read_csv(f"{DATA_DIR}/data_clinical_patient.txt", sep="\t", comment="#")
sample  = pd.read_csv(f"{DATA_DIR}/data_clinical_sample.txt", sep="\t", comment="#")

print(sample.columns.tolist())
print(sample["CANCER_TYPE"].value_counts().head(20))

['SAMPLE_ID', 'PATIENT_ID', 'GLEASON_SAMPLE_LEVEL', 'PDL1_POSITIVE', 'CANCER_TYPE', 'SAMPLE_TYPE', 'SAMPLE_CLASS', 'METASTATIC_SITE', 'PRIMARY_SITE', 'CANCER_TYPE_DETAILED', 'GENE_PANEL', 'SAMPLE_COVERAGE', 'TUMOR_PURITY', 'ONCOTREE_CODE', 'MSI_COMMENT', 'MSI_SCORE', 'MSI_TYPE', 'SOMATIC_STATUS', 'CLINICAL_GROUP', 'PATHOLOGICAL_GROUP', 'CLINICAL_SUMMARY', 'ICD_O_HISTOLOGY_DESCRIPTION', 'DIAGNOSIS_DESCRIPTION', 'TMB_NONSYNONYMOUS']
CANCER_TYPE
Non-Small Cell Lung Cancer    7809
Colorectal Cancer             5543
Breast Cancer                 5368
Prostate Cancer               3211
Pancreatic Cancer             3109
Name: count, dtype: int64


In [64]:
nsclc_samples = sample[sample["CANCER_TYPE"] == "Non-Small Cell Lung Cancer"]
nsclc_patient_ids = set(nsclc_samples["PATIENT_ID"].unique())

nsclc_patients = patient[patient["PATIENT_ID"].isin(nsclc_patient_ids)]

print(len(nsclc_patient_ids), "NSCLC patients")

nsclc_patients.to_csv(f"{root_path}/nsclc_clinical_patient.csv", index=False)
nsclc_samples.to_csv(f"{root_path}/nsclc_clinical_sample.csv", index=False)  

7809 NSCLC patients


In [65]:
from pathlib import Path

# Define directory path
dir_path = Path(root_path)

# Create the directory
# parents=True creates any missing parent directories
# exist_ok=True prevents raising an error if the directory already exists
dir_path.mkdir(parents=True, exist_ok=True)

print(f"Directory '{dir_path}' created successfully!")
timeline_files = [
    "data_timeline_diagnosis.txt",
    "data_timeline_treatment.txt",
    "data_timeline_tumor_sites.txt",
    "data_timeline_cancer_presence.txt",
    "data_timeline_progression.txt",
    "data_timeline_performance_status.txt",
    "data_timeline_pdl1.txt",
]

for fname in timeline_files:
    df = pd.read_csv(f"{DATA_DIR}/{fname}", sep="\t", comment="#")
    filtered = df[df["PATIENT_ID"].isin(nsclc_patient_ids)]
    out_name = f"{root_path}/nsclc_" + fname
    filtered.to_csv(out_name, index=False)
    print(fname, "→", len(df), "rows total,", len(filtered), "rows kept")

Directory 'datasets\msk_chord_2024\msk_chord_out' created successfully!
data_timeline_diagnosis.txt → 25145 rows total, 7927 rows kept


data_timeline_treatment.txt → 134943 rows total, 28923 rows kept
data_timeline_tumor_sites.txt → 506647 rows total, 177385 rows kept
data_timeline_cancer_presence.txt → 398528 rows total, 142260 rows kept
data_timeline_progression.txt → 398874 rows total, 142189 rows kept
data_timeline_performance_status.txt → 157674 rows total, 46076 rows kept
data_timeline_pdl1.txt → 6368 rows total, 5864 rows kept


In [66]:
mutations = pd.read_csv(f"{DATA_DIR}/data_mutations.txt", sep="\t", comment="#", low_memory=False)
print(mutations.columns.tolist())  # confirm exact barcode column name before filtering

nsclc_sample_ids = set(nsclc_samples["SAMPLE_ID"].unique())
nsclc_mutations = mutations[mutations["Tumor_Sample_Barcode"].isin(nsclc_sample_ids)]
nsclc_mutations.to_csv(f"{root_path}/nsclc_mutations.csv", index=False)
print(len(nsclc_mutations), "mutation records for NSCLC")

['Hugo_Symbol', 'Entrez_Gene_Id', 'Center', 'NCBI_Build', 'Chromosome', 'Start_Position', 'End_Position', 'Strand', 'Consequence', 'Variant_Classification', 'Variant_Type', 'Reference_Allele', 'Tumor_Seq_Allele1', 'Tumor_Seq_Allele2', 'dbSNP_RS', 'dbSNP_Val_Status', 'Tumor_Sample_Barcode', 'Matched_Norm_Sample_Barcode', 'Match_Norm_Seq_Allele1', 'Match_Norm_Seq_Allele2', 'Tumor_Validation_Allele1', 'Tumor_Validation_Allele2', 'Match_Norm_Validation_Allele1', 'Match_Norm_Validation_Allele2', 'Verification_Status', 'Validation_Status', 'Mutation_Status', 'Sequencing_Phase', 'Sequence_Source', 'Validation_Method', 'Score', 'BAM_File', 'Sequencer', 't_ref_count', 't_alt_count', 'n_ref_count', 'n_alt_count', 'HGVSc', 'HGVSp', 'HGVSp_Short', 'Transcript_ID', 'RefSeq', 'Protein_position', 'Codons', 'Exon_Number', 'COMMENTS', 'PATH_SCORE', 'AA_MAF', 'AFR_MAF', 'ALLELE_NUM', 'AMR_MAF', 'ASN_MAF', 'Allele', 'Amino_Acid_Change', 'Amino_acids', 'BIOTYPE', 'CANONICAL', 'CCDS', 'CDS_position', 'CLIN

In [67]:
print(nsclc_samples["CANCER_TYPE_DETAILED"].value_counts())
print(nsclc_samples["ONCOTREE_CODE"].value_counts())

CANCER_TYPE_DETAILED
Lung Adenocarcinoma                                 5957
Lung Squamous Cell Carcinoma                         822
Non-Small Cell Lung Cancer                           499
Large Cell Neuroendocrine Carcinoma                  134
Lung Carcinoid                                        91
Poorly Differentiated Non-Small Cell Lung Cancer      77
Lung Adenosquamous Carcinoma                          56
Pleomorphic Carcinoma of the Lung                     45
Atypical Lung Carcinoid                               43
Lung Neuroendocrine Tumor                             32
Sarcomatoid Carcinoma of the Lung                     19
Adenoid Cystic Carcinoma of the Lung                   7
Large Cell Lung Carcinoma                              7
Lung Adenocarcinoma In Situ                            6
Lymphoepithelioma-like Carcinoma of the Lung           5
Spindle Cell Carcinoma of the Lung                     4
Mucoepidermoid Carcinoma of the Lung                   2
Basaloid L

In [68]:
# Check for patients with multiple samples first — this determines merge strategy
sample_counts = nsclc_samples.groupby("PATIENT_ID").size()
print("Patients with >1 sample:", (sample_counts > 1).sum(), "out of", len(sample_counts))

# Most MSK-CHORD patients have exactly 1 sample (per SAMPLE_COUNT field seen earlier).
# For patients with multiple samples, keep the sample with the highest TUMOR_PURITY 
# (or swap to "most recent" if you have a date field you prefer) as the representative sample.
nsclc_samples_dedup = (
    nsclc_samples.sort_values("TUMOR_PURITY", ascending=False)
    .drop_duplicates(subset="PATIENT_ID", keep="first")
)

# Merge patient-level clinical data with (deduplicated) sample-level molecular data
merged = nsclc_patients.merge(
    nsclc_samples_dedup, on="PATIENT_ID", how="left", suffixes=("_patient", "_sample")
)
print(merged.shape, "→ one row per patient")

Patients with >1 sample: 0 out of 7809
(7809, 49) → one row per patient


In [70]:
for f in [f"{root_path}/nsclc_data_timeline_tumor_sites.txt", f"{root_path}/nsclc_data_timeline_performance_status.txt", f"{root_path}/nsclc_data_timeline_pdl1.txt"]:
    df = pd.read_csv(f, nrows=3)
    print(f, "→", df.columns.tolist())
    print(df.head(2), "\n")

datasets/msk_chord_2024/msk_chord_out/nsclc_data_timeline_tumor_sites.txt → ['PATIENT_ID', 'START_DATE', 'STOP_DATE', 'EVENT_TYPE', 'SUBTYPE', 'SOURCE', 'SOURCE_SPECIFIC', 'TUMOR_SITE', 'CHEST', 'ABDOMEN', 'PELVIS', 'HEAD', 'OTHER']
  PATIENT_ID  START_DATE  STOP_DATE EVENT_TYPE      SUBTYPE  \
0  P-0001340         -31        NaN  Diagnosis  Tumor Sites   
1  P-0001340          49        NaN  Diagnosis  Tumor Sites   

                    SOURCE SOURCE_SPECIFIC      TUMOR_SITE  CHEST  ABDOMEN  \
0  Radiology Reports (NLP)             PET  Adrenal Glands      0        0   
1  Radiology Reports (NLP)              CT  Adrenal Glands      1        1   

   PELVIS  HEAD  OTHER  
0       0     0      1  
1       0     0      0   

datasets/msk_chord_2024/msk_chord_out/nsclc_data_timeline_performance_status.txt → ['PATIENT_ID', 'START_DATE', 'STOP_DATE', 'EVENT_TYPE', 'SUBTYPE', 'STYLE_COLOR', 'ECOG']
  PATIENT_ID  START_DATE  STOP_DATE EVENT_TYPE             SUBTYPE  \
0  P-0000012         1

In [71]:
# Tumor sites: number of distinct metastatic sites ever recorded (unchanged — column was already correct)
tumor_sites_summary = (
    nsclc_tumor_sites.groupby("PATIENT_ID")["TUMOR_SITE"]
    .nunique()
    .reset_index(name="num_distinct_tumor_sites")
)

# Performance status: latest *non-null* ECOG score per patient
perf_status_latest = (
    nsclc_perf_status.dropna(subset=["ECOG"])
    .sort_values("START_DATE")
    .groupby("PATIENT_ID")
    .tail(1)[["PATIENT_ID", "ECOG"]]
    .rename(columns={"ECOG": "latest_ecog_status"})
)

# PD-L1: ever positive? (column name was already correct)
pdl1_summary = (
    nsclc_pdl1.groupby("PATIENT_ID")["PDL1_POSITIVE"]
    .apply(lambda x: (x == "Yes").any())
    .reset_index(name="ever_pdl1_positive")
)

# Fold everything into the main merged table
for summary_df in [treatment_summary, tumor_sites_summary, perf_status_latest, pdl1_summary]:
    merged = merged.merge(summary_df, on="PATIENT_ID", how="left")

print(merged.shape)
merged.to_csv(f"{root_path}/nsclc_patient_level_features.csv", index=False)

(7809, 53)


In [72]:
# 1. Missingness check on the new summary columns — flagged this as worth knowing
print(merged[["num_treatment_events", "num_distinct_tumor_sites", "latest_ecog_status", "ever_pdl1_positive"]].isna().sum())

# 2. Quick sanity look at value ranges
print(merged["num_treatment_events"].describe())
print(merged["latest_ecog_status"].value_counts(dropna=False))
print(merged["ever_pdl1_positive"].value_counts(dropna=False))

# 3. Confirm no duplicate patients slipped through the sample dedup
print(merged["PATIENT_ID"].duplicated().sum(), "duplicate patient rows (should be 0)")

num_treatment_events        1698
num_distinct_tumor_sites     384
latest_ecog_status          7809
ever_pdl1_positive          3395
dtype: int64
count    6111.000000
mean        4.732941
std         3.478721
min         1.000000
25%         2.000000
50%         4.000000
75%         6.000000
max        33.000000
Name: num_treatment_events, dtype: float64
latest_ecog_status
NaN    7809
Name: count, dtype: int64
ever_pdl1_positive
NaN      3395
True     2981
False    1433
Name: count, dtype: int64
0 duplicate patient rows (should be 0)


In [73]:
print(merged[["num_treatment_events", "num_distinct_tumor_sites", "latest_ecog_status", "ever_pdl1_positive"]].isna().sum())
print()
print(merged["num_treatment_events"].describe())
print()
print(merged["latest_ecog_status"].value_counts(dropna=False))
print()
print(merged["ever_pdl1_positive"].value_counts(dropna=False))
print()
print(merged["PATIENT_ID"].duplicated().sum(), "duplicate patient rows (should be 0)")
print()
# Also check the two candidate target columns for missingness/class balance
print(merged["CANCER_TYPE_DETAILED"].isna().sum(), "missing CANCER_TYPE_DETAILED")
print(merged["STAGE_HIGHEST_RECORDED"].value_counts(dropna=False))

num_treatment_events        1698
num_distinct_tumor_sites     384
latest_ecog_status          7809
ever_pdl1_positive          3395
dtype: int64

count    6111.000000
mean        4.732941
std         3.478721
min         1.000000
25%         2.000000
50%         4.000000
75%         6.000000
max        33.000000
Name: num_treatment_events, dtype: float64

latest_ecog_status
NaN    7809
Name: count, dtype: int64

ever_pdl1_positive
NaN      3395
True     2981
False    1433
Name: count, dtype: int64

0 duplicate patient rows (should be 0)

0 missing CANCER_TYPE_DETAILED
STAGE_HIGHEST_RECORDED
Stage 1-3    4387
Stage 4      3421
Unknown         1
Name: count, dtype: int64


In [74]:
# OPTION A: Subtype classification target (diagnosis-style — "what kind of NSCLC is this")
# Collapse rare subtypes into "Other" so classes aren't wildly imbalanced
top_subtypes = merged["CANCER_TYPE_DETAILED"].value_counts().nlargest(4).index.tolist()
merged["subtype_target"] = merged["CANCER_TYPE_DETAILED"].apply(
    lambda x: x if x in top_subtypes else "Other"
)
print(merged["subtype_target"].value_counts())

# OPTION B: Stage-at-diagnosis target (screening/staging-style — "how advanced is it")
merged["stage_target"] = merged["STAGE_HIGHEST_RECORDED"].fillna("Unknown")
print(merged["stage_target"].value_counts())

subtype_target
Lung Adenocarcinoma                    5957
Lung Squamous Cell Carcinoma            822
Non-Small Cell Lung Cancer              499
Other                                   397
Large Cell Neuroendocrine Carcinoma     134
Name: count, dtype: int64
stage_target
Stage 1-3    4387
Stage 4      3421
Unknown         1
Name: count, dtype: int64


In [75]:
!pip install -U scikit-learn


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [76]:
from sklearn.model_selection import train_test_split

TARGET_COL = "subtype_target"  # switch to "stage_target" if that's the actual task

# Drop rows with unknown/missing target — can't train or evaluate on these
model_df = merged[merged[TARGET_COL] != "Unknown"].copy() if TARGET_COL == "stage_target" else merged.copy()

train_df, temp_df = train_test_split(
    model_df, test_size=0.30, stratify=model_df[TARGET_COL], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df[TARGET_COL], random_state=42
)

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)
print("\nTrain class balance:\n", train_df[TARGET_COL].value_counts(normalize=True))
print("\nTest class balance:\n", test_df[TARGET_COL].value_counts(normalize=True))

train_df.to_csv(f"{root_path}/nsclc_train.csv", index=False)
val_df.to_csv(f"{root_path}/nsclc_val.csv", index=False)
test_df.to_csv(f"{root_path}/nsclc_test.csv", index=False)

Train: (5466, 55) Val: (1171, 55) Test: (1172, 55)

Train class balance:
 subtype_target
Lung Adenocarcinoma                    0.762898
Lung Squamous Cell Carcinoma           0.105196
Non-Small Cell Lung Cancer             0.063849
Other                                  0.050860
Large Cell Neuroendocrine Carcinoma    0.017197
Name: proportion, dtype: float64

Test class balance:
 subtype_target
Lung Adenocarcinoma                    0.762799
Lung Squamous Cell Carcinoma           0.105802
Non-Small Cell Lung Cancer             0.063993
Other                                  0.050341
Large Cell Neuroendocrine Carcinoma    0.017065
Name: proportion, dtype: float64


In [77]:
# Check raw ECOG data before any filtering
print(nsclc_perf_status["ECOG"].notna().sum(), "non-null ECOG values out of", len(nsclc_perf_status))
print(nsclc_perf_status["ECOG"].unique()[:20])

# Check what perf_status_latest actually contains before the merge
print(perf_status_latest.shape)
print(perf_status_latest.head())

0 non-null ECOG values out of 46076
[nan]
(0, 2)
Empty DataFrame
Columns: [PATIENT_ID, latest_ecog_status]
Index: []


In [78]:
merged["pdl1_status_category"] = merged["ever_pdl1_positive"].map({True: "Positive", False: "Negative"}).fillna("Not Tested")

In [79]:
# Check the FULL unfiltered performance status file, not just the NSCLC subset
full_perf_status = pd.read_csv(f"{DATA_DIR}/data_timeline_performance_status.txt", sep="\t", comment="#")
print(full_perf_status["ECOG"].notna().sum(), "non-null ECOG values out of", len(full_perf_status))

# Also check the other non-key column in that file — STYLE_COLOR might be where
# performance status is actually encoded (e.g. as a color-coded severity level)
# rather than in a numeric ECOG field
print(full_perf_status["STYLE_COLOR"].value_counts(dropna=False).head(20))
print(full_perf_status["SUBTYPE"].value_counts(dropna=False))

0 non-null ECOG values out of 157674
STYLE_COLOR
NaN    157674
Name: count, dtype: int64
SUBTYPE
Performance Status    157674
Name: count, dtype: int64
